# Pengujian CV Klasik: Color-LBP Tanpa Kanal Y dan V (4 Kanal Krominans)

- **Gatekeeper 1 (Liveness):** `Cr, Cb` + `H, S` saja — tanpa luminansi `Y` & `V` (4 kanal = **1.024 dimensi**, 1-NN Chi-Square `d_real < d_spoof`)
- **Gatekeeper 2 (Identitas):** Grayscale `100x100` -> `equalizeHist` -> LBPH Grid `8x8` (16.384 dimensi, `Chi2 <= 23.0`)
- **Kamera Live:** Jalankan Sel 4, lalu tekan **`q`** pada jendela kamera untuk berhenti.

In [1]:
import json
from pathlib import Path
import cv2
import numpy as np

haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
LBPH_THRESHOLD = 23.0

def deteksi_wajah(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = haar.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) == 0:
        return None, None
    bx, by, bw, bh = max(faces, key=lambda f: f[2] * f[3])
    return frame_bgr[by:by+bh, bx:bx+bw], (int(bx), int(by), int(bw), int(bh))

def ekstrak_lbp(gray_img, grid_x=1, grid_y=1):
    h, w = gray_img.shape
    center = gray_img[1:h-1, 1:w-1]
    lbp = np.zeros_like(center, dtype=np.uint8)
    shifts = [(-1,-1,0), (-1,0,1), (-1,1,2), (0,1,3), (1,1,4), (1,0,5), (1,-1,6), (0,-1,7)]
    for dy, dx, bit in shifts:
        nb = gray_img[1+dy:h-1+dy, 1+dx:w-1+dx]
        lbp |= ((nb >= center).astype(np.uint8) << bit)
    hists = []
    for row in np.array_split(lbp, grid_y, axis=0):
        for col in np.array_split(row, grid_x, axis=1):
            hist, _ = np.histogram(col, bins=256, range=(0, 256))
            hist = hist.astype(np.float32)
            hist /= (hist.sum() + 1e-7)
            hists.append(hist)
    return np.concatenate(hists)

def ekstrak_color_lbp_tanpa_yv(face_bgr):
    """4 kanal krominans tanpa Y & V: Cr, Cb (YCrCb) + H, S (HSV) -> 1.024 dimensi."""
    face = cv2.resize(face_bgr, (128, 128))
    ycrcb = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)
    hsv = cv2.cvtColor(face, cv2.COLOR_BGR2HSV)
    fitur = [
        ekstrak_lbp(ycrcb[:, :, 1]),  # Cr (tanpa Y)
        ekstrak_lbp(ycrcb[:, :, 2]),  # Cb
        ekstrak_lbp(hsv[:, :, 0]),    # H
        ekstrak_lbp(hsv[:, :, 1]),    # S (tanpa V)
    ]
    return np.concatenate(fitur).astype(np.float32)

def chi2_min(mat, vec):
    return float(np.min(0.5 * np.sum(((mat - vec) ** 2) / (mat + vec + 1e-7), axis=1)))

print("Setup 4 kanal tanpa Y & V (Cr, Cb, H, S) siap.")

Setup 4 kanal tanpa Y & V (Cr, Cb, H, S) siap.


In [2]:
# 1. Bangun prototipe Liveness 4 kanal dari sumber real & spoof yang sama dengan proyek
proto = json.loads(Path("api_data/color_lbp_proto.json").read_text(encoding="utf-8"))
real_vecs, spoof_vecs = [], []

for p in proto["real_sources"]:
    img = cv2.imread(p)
    if img is None: continue
    crop, _ = deteksi_wajah(img)
    real_vecs.append(ekstrak_color_lbp_tanpa_yv(crop if crop is not None else img))

for p in proto["spoof_sources"]:
    img = cv2.imread(p)
    if img is None: continue
    crop, _ = deteksi_wajah(img)
    spoof_vecs.append(ekstrak_color_lbp_tanpa_yv(crop if crop is not None else img))

REAL_MAT = np.asarray(real_vecs, dtype=np.float32)
SPOOF_MAT = np.asarray(spoof_vecs, dtype=np.float32)

# 2. Muat database identitas LBPH (tanpa Eye Alignment)
lbph_raw = json.loads(Path("api_data/faces_lbph.json").read_text(encoding="utf-8"))
LBPH_DB = {nama: np.asarray(data["exemplars"], dtype=np.float32) for nama, data in lbph_raw.items()}

print(f"Prototipe Liveness tanpa Y & V : Real={REAL_MAT.shape}, Spoof={SPOOF_MAT.shape}")
print(f"Database LBPH Identitas        : { {k: len(v) for k, v in LBPH_DB.items()} }")

Prototipe Liveness tanpa Y & V : Real=(75, 1024), Spoof=(30, 1024)
Database LBPH Identitas        : {'Faza': 50, 'Raffin': 50, 'Rafi': 48}


In [3]:
def uji_gambar(frame_bgr, fallback_full=False):
    crop, box = deteksi_wajah(frame_bgr)
    if crop is None:
        if not fallback_full:
            return None
        crop = frame_bgr
    # Gatekeeper 1: Liveness (4 kanal krominans tanpa Y & V)
    v_live = ekstrak_color_lbp_tanpa_yv(crop)
    d_real = chi2_min(REAL_MAT, v_live)
    d_spoof = chi2_min(SPOOF_MAT, v_live)
    is_real = d_real < d_spoof

    # Gatekeeper 2: Identitas (LBPH Grayscale 100x100)
    nama_cocok, chi2_id = None, 999.0
    if is_real:
        gray_100 = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), (100, 100))
        eq = cv2.equalizeHist(gray_100)
        v_id = ekstrak_lbp(eq, grid_x=8, grid_y=8)
        for nama, ex in LBPH_DB.items():
            d = chi2_min(ex, v_id)
            if d < chi2_id:
                chi2_id, nama_cocok = d, nama
    granted = is_real and (chi2_id <= LBPH_THRESHOLD)
    return {"box": box, "is_real": is_real, "d_real": d_real, "d_spoof": d_spoof,
            "nama": nama_cocok if granted else "Unknown", "chi2": chi2_id, "granted": granted}

# Uji cepat offline: 20 Real per penghuni + Spoof di luar data latih
spoof_latih = set(proto["spoof_sources"])
uji_real = [p for org in ["Faza", "Raffin", "Rafy"] for p in sorted((Path("dataset baru") / org).glob("*.jpg"))[:20]]
uji_spoof = [p for p in sorted(Path("spoof").glob("*.*")) if str(p).replace("/", "\\") not in spoof_latih][:60]

res_real = [uji_gambar(cv2.imread(str(p)), fallback_full=True) for p in uji_real]
res_spoof = [uji_gambar(cv2.imread(str(p)), fallback_full=True) for p in uji_spoof]
real_ok = sum(1 for r in res_real if r["is_real"])
spoof_ok = sum(1 for r in res_spoof if not r["is_real"])
print("Uji Offline [Tanpa Y & V - 4 Kanal Krominans]:")
print(f"  Real Lulus    : {real_ok}/{len(res_real)} ({real_ok/len(res_real)*100:.1f}%)")
print(f"  Spoof Ditolak : {spoof_ok}/{len(res_spoof)} ({spoof_ok/len(res_spoof)*100:.1f}%)")

Uji Offline [Tanpa Y & V - 4 Kanal Krominans]:
  Real Lulus    : 58/60 (96.7%)
  Spoof Ditolak : 59/60 (98.3%)


In [7]:
# Test Kamera Real-Time + Capture (tekan SPACE / 'c' untuk capture, 'q' untuk keluar)
captured_frame = None
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka!")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        raw_frame = frame.copy()
        res = uji_gambar(frame)
        if res is not None:
            x, y, w, h = res["box"]
            if not res["is_real"]:
                warna = (0, 0, 255)
                teks = f"SPOOF | R:{res['d_real']:.3f} S:{res['d_spoof']:.3f}"
            elif res["granted"]:
                warna = (0, 200, 0)
                teks = f"REAL: {res['nama']} | Chi2:{res['chi2']:.1f}"
            else:
                warna = (0, 165, 255)
                teks = f"REAL: Unknown | Chi2:{res['chi2']:.1f}"

            cv2.rectangle(frame, (x, y), (x + w, y + h), warna, 2)
            cv2.putText(frame, teks, (x, max(25, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, warna, 2)
            cv2.putText(frame, f"4 Kanal Tanpa Y & V | d_real={res['d_real']:.3f} d_spoof={res['d_spoof']:.3f}",
                        (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        else:
            cv2.putText(frame, "Wajah tidak terdeteksi", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.putText(frame, "SPACE / C: Capture ke Subplot | Q: Keluar", (10, frame.shape[0] - 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)
        cv2.imshow("Test Kamera - Tanpa Y & V (4 Kanal)", frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("c"), ord("C"), 32):  # SPACE atau C untuk capture
            captured_frame = raw_frame
            print("Frame berhasil di-capture! Jalankan sel berikutnya untuk melihat subplot step-by-step.")
            break
        if key in (ord("q"), 27):
            captured_frame = raw_frame
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# Sel 5: Tampilkan hasil capture step-by-step dalam subplot (otomatis pakai sample jika belum capture)
frame_uji = captured_frame if ("captured_frame" in globals() and captured_frame is not None) else cv2.imread(str(uji_real[0]))

crop, box = deteksi_wajah(frame_uji)
if crop is None:
    raise RuntimeError("Wajah tidak terdeteksi pada frame yang di-capture. Ulangi capture.")

x, y, w, h = box
img_haar = frame_uji.copy()
cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 220, 0), 3)

face_128 = cv2.resize(crop, (128, 128))
ycrcb   = cv2.cvtColor(face_128, cv2.COLOR_BGR2YCrCb)
hsv     = cv2.cvtColor(face_128, cv2.COLOR_BGR2HSV)

# Normalisasi Cr & Cb agar rentang 0-255 kelihatan di colormap
cr_vis = cv2.normalize(ycrcb[:,:,1], None, 0, 255, cv2.NORM_MINMAX)
cb_vis = cv2.normalize(ycrcb[:,:,2], None, 0, 255, cv2.NORM_MINMAX)

gray_100 = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), (100, 100))
eq_100   = cv2.equalizeHist(gray_100)

# Hitung peta LBP 98x98 untuk visualisasi
c_px    = eq_100[1:99, 1:99]
lbp_map = np.zeros_like(c_px, dtype=np.uint8)
for dy, dx, bit in [(-1,-1,0),(-1,0,1),(-1,1,2),(0,1,3),(1,1,4),(1,0,5),(1,-1,6),(0,-1,7)]:
    lbp_map |= ((eq_100[1+dy:99+dy, 1+dx:99+dx] >= c_px).astype(np.uint8) << bit)

# Histogram NYATA dari peta LBP (distribusi 0-255)
hist_vals, _ = np.histogram(lbp_map, bins=64, range=(0, 256))

res = uji_gambar(frame_uji)

# Buat gambar grafik histogram pakai OpenCV
hist_canvas = np.full((220, 270, 3), 245, dtype=np.uint8)
cv2.putText(hist_canvas, "LBP Histogram", (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (60,60,60), 1)
mx = float(hist_vals.max()) if hist_vals.max() > 0 else 1.0
bar_color = (50, 170, 50) if res["is_real"] else (30, 30, 200)
for i, val in enumerate(hist_vals):
    bh = int((float(val) / mx) * 155)
    cv2.rectangle(hist_canvas, (4 + i*4, 195 - bh), (6 + i*4, 196), bar_color, -1)
cv2.line(hist_canvas, (4, 196), (262, 196), (100,100,100), 1)
lbl = f"{'REAL' if res['is_real'] else 'SPOOF'} | {res['nama']}"
lbl2 = f"Chi2={res['chi2']:.2f} | {'AKSES OK' if res['granted'] else 'DITOLAK'}"
clr = (20, 130, 20) if res["granted"] else (0, 0, 200)
cv2.putText(hist_canvas, lbl,  (5, 212), cv2.FONT_HERSHEY_SIMPLEX, 0.38, clr, 1)
cv2.putText(hist_canvas, lbl2, (5, 219), cv2.FONT_HERSHEY_SIMPLEX, 0.33, clr, 1)

langkah = [
    ("1. Input + Haar",       img_haar,      None),
    ("2. ROI Crop (BGR)",     face_128,       None),
    ("3a. Kanal Cr (YCrCb)", cr_vis,         cv2.COLORMAP_JET),
    ("3b. Kanal Cb (YCrCb)", cb_vis,         cv2.COLORMAP_OCEAN),
    ("3c. Kanal H (HSV)",    hsv[:,:,0],     cv2.COLORMAP_HSV),
    ("3d. Kanal S (HSV)",    hsv[:,:,1],     cv2.COLORMAP_MAGMA),
    ("4. Equalize 100x100",  eq_100,         "gray"),
    ("5. Peta Tekstur LBP",  lbp_map,        "gray"),
    ("6. LBP Histogram",     hist_canvas,    None),
]

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
    fig.suptitle("Pipeline 4-Kanal: Krominans Saja (Cr,Cb,H,S)", fontsize=13, fontweight="bold")
    for ax, (judul, gbr, cmap) in zip(axes.flat, langkah):
        if isinstance(cmap, int):
            gbr_c = cv2.applyColorMap(gbr, cmap)
            ax.imshow(cv2.cvtColor(gbr_c, cv2.COLOR_BGR2RGB))
        elif gbr.ndim == 2:
            ax.imshow(gbr, cmap="gray")
        elif gbr.ndim == 3 and gbr.shape[2] == 3:
            ax.imshow(cv2.cvtColor(gbr, cv2.COLOR_BGR2RGB))
        ax.set_title(judul, fontsize=9, fontweight="bold")
        ax.axis("off")
    for ax in axes.flat[len(langkah):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("hasil_subplot_tanpa_yv.png", dpi=110, bbox_inches="tight")
    plt.show()
    print("Subplot tersimpan: hasil_subplot_tanpa_yv.png")
except ImportError:
    # Fallback: susun dengan OpenCV tile
    TILE = 220
    rows, cols = 2, 5
    canvas = np.full((rows * TILE, cols * TILE, 3), 230, dtype=np.uint8)
    for idx, (judul, gbr, cmap) in enumerate(langkah):
        r, c = divmod(idx, cols)
        if isinstance(cmap, int):
            tile = cv2.applyColorMap(gbr, cmap)
        elif gbr.ndim == 2:
            tile = cv2.cvtColor(gbr, cv2.COLOR_GRAY2BGR)
        else:
            tile = gbr.copy()
        tile = cv2.resize(tile, (TILE - 10, TILE - 36))
        y0 = r * TILE + 30; x0 = c * TILE + 5
        canvas[y0:y0 + TILE - 36, x0:x0 + TILE - 10] = tile
        cv2.putText(canvas, judul, (x0, r * TILE + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (20,20,20), 1)
    cv2.imwrite("hasil_subplot_tanpa_yv.png", canvas)
    from IPython.display import Image, display
    display(Image("hasil_subplot_tanpa_yv.png"))
    print("Subplot tersimpan: hasil_subplot_tanpa_yv.png (OpenCV fallback)")
